# 📓 Llama-Index Practice

In this section, we will create a simple Llama Index app and learn how to log and get feedback on an LLM response. Additionally, we will explore how the query engine can be modified based on its intended purpose.​

Through this process, we hope to gain a clear understanding of the role and functionalities of the Llama Index, as well as the specific inputs and outputs of each method.  

<img src="https://miro.medium.com/v2/resize:fit:1400/0*CrywD0tloiK9dgy_.png" width="600">

This section is divided into three main stages:

###I. Build a Query Engine  
###II. Use Database Management Method  
###III. Make a Custom Query Engine

Let's first look at what is needed to build a Query Engine.

## I. Buliding a Query Engine

The task of this section is to create a Query Engine that takes a user's query, retrieves for related content, and returns a final summary.  

Ignore the `Vector DB` in the picture. We don't use external vector database in the pracetice. Instead, we will use local memory as simple vector store.


<img src="https://i.imgur.com/bR4xaBd.png">

To accomplish this task, the following steps will be taken.
<br/>

#### 1. Install and Import Libraries
#### 2. DownLoad Data
#### 3. Make Query Engine from Index Directly
#### 4. Conduct Activity


### 1. Install and Import Libraries

In [1]:
### YOUR CODE HERE ###
import os
os.environ.pop("PYTHONPATH", None)

! pip install llama_index==0.12.2 openai==1.55.3 packaging>=24.0 streamlit==1.35.0 --quiet

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
onnx 1.23.2 requires protobuf>=6.31.1, but you have protobuf 4.25.9 which is incompatible.
opentelemetry-proto 1.36.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.9 which is incompatible.


In [54]:
### YOUR CODE HERE ###

import os
### My Code ###
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')
assert os.getenv('OPENAI_API_KEY'),  "OPENAI_API_KEY가 로드되지 않았습니다."
### My Code End ###

#os.environ["OPENAI_API_KEY"] = "sk-..." #Insert your openai api key


from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Document, PromptTemplate
from openai import OpenAI
from llama_index.llms.openai import OpenAI as LlamaOpenAI

from llama_index.core.query_engine import CustomQueryEngine
from llama_index.core.retrievers import BaseRetriever
from llama_index.core import get_response_synthesizer
from llama_index.core.response_synthesizers import BaseSynthesizer

import textwrap
import openai

### 2. DownLoad Data

Previously, we learned that the Query Engine receives the following elements as input and output.
<br/>
#### Input: **User Query**, **Vector Database**
#### Output: **Summary of Retrieved Passages**
<br/>

Here, the User Query is an element that we can decide. However, since the Vector Database must contain information related to the Query, we need to consider how to configure the Vector Database before proceeding with the practice.

The Vector Database can be divided into two categories: a storage that allows the query engine to retrieve related passages, and the data containing the related passages. In this practice, we will use local memory as a data storage instead of an external Vector Database, so the Vector Database will be referred to as a Vector Store from now on. For the data, we will use a simple txt file provided by Llama Index.

Before we begin the practical exercises, let's first take a look at the txt file we used. The code below can be used to download the txt file we used with the `wget` command.


In [3]:
### YOUR CODE HERE ###

import os, urllib.request

os.makedirs("data", exist_ok=True)
url = "https://raw.githubusercontent.com/run-llama/llama_index/main/docs/examples/data/paul_graham/paul_graham_essay.txt"
urllib.request.urlretrieve(url, "data/paul_graham_essay.txt")

('data/paul_graham_essay.txt', <http.client.HTTPMessage at 0x1f34f56a210>)

If there are no issues with the download, you should be able to see that the `paul_graham_essay.txt` file has been downloaded.

This example uses the text of Paul Graham’s essay, [“What I Worked On”](https://paulgraham.com/worked.html), and is the canonical llama-index example.


Let’s print the first fifteen lines of the txt file. This will allow us to see that the text file deals with the experiences and reflections of one person's life.


In [4]:
import os
print(os.getcwd())

c:\AI_Workspace\ai-speciallist-exam\rag\실습자료\1일차\Code


In [17]:
### YOUR CODE HERE ###
path_to_txt = os.path.join(os.getcwd(), 'data', 'paul_graham_essay.txt')
#path_to_txt = f'{os.getcwd()}\data\paul_graham_essay.txt' #change this path

with open(path_to_txt, 'r', encoding='utf-8') as file:
    for i in range(15):
        line = file.readline()
        if not line:
            break
        print(line.strip())
print("\n...")



What I Worked On

February 2021

Before college the two main things I worked on, outside of school, were writing and programming. I didn't write essays. I wrote what beginning writers were supposed to write then, and probably still are: short stories. My stories were awful. They had hardly any plot, just characters with strong feelings, which I imagined made them deep.

The first programs I tried writing were on the IBM 1401 that our school district used for what was then called "data processing." This was in 9th grade, so I was 13 or 14. The school district's 1401 happened to be in the basement of our junior high school, and my friend Rich Draves and I got permission to use it. It was like a mini Bond villain's lair down there, with all these alien-looking machines — CPU, disk drives, printer, card reader — sitting up on a raised floor under bright fluorescent lights.

The language we used was an early version of Fortran. You had to type programs on punch cards, then stack them in t

### 3. Make Query Engine from Index Directly

Now let's create a query engine unsing the above information.

First, since Llama Index cannot directly handle text files in their raw form, we must convert them into Documents and Nodes. This can be accomplished with the following code.

```Python
documents = SimpleDirectoryReader("data").load_data()
```

SimpleDirectoryReader takes the directory path containing data that will go into the Vector Store as input and converts all the files within that directory into Document objects. The data types that can be converted into Documents are as follows.


*   .txt - text file  
*   .csv - comma-separated values  
*   .docx - Microsoft Word  
*   .epub - EPUB ebook format  
*   .hwp - Hangul Word Processor  
*   .ipynb - Jupyter Notebook  
*   .jpeg, .jpg - JPEG image  
*   .mbox - MBOX email archive  
*   .md - Markdown  
*   .mp3, .mp4 - audio and video  
*   .pdf - Portable Document Format  
*   .png - Portable Network Graphics  
*   .ppt, .pptm, .pptx - Microsoft PowerPoint  

You can view the text contained in the `documents` using the following command:

```Python
for i in range(len(documents)):
  print(documents[i].text)
```

In [18]:
### YOUR CODE HERE ###

documents = SimpleDirectoryReader("data").load_data()

In [19]:
### YOUR CODE HERE ###

for i in range(len(documents)):
  print(documents[i].text)



What I Worked On

February 2021

Before college the two main things I worked on, outside of school, were writing and programming. I didn't write essays. I wrote what beginning writers were supposed to write then, and probably still are: short stories. My stories were awful. They had hardly any plot, just characters with strong feelings, which I imagined made them deep.

The first programs I tried writing were on the IBM 1401 that our school district used for what was then called "data processing." This was in 9th grade, so I was 13 or 14. The school district's 1401 happened to be in the basement of our junior high school, and my friend Rich Draves and I got permission to use it. It was like a mini Bond villain's lair down there, with all these alien-looking machines — CPU, disk drives, printer, card reader — sitting up on a raised floor under bright fluorescent lights.

The language we used was an early version of Fortran. You had to type programs on punch cards, then stack them in t

Once the Document is complete, we need to use this data to create an index.

The index is created using the VectorStoreIndex.from_documents() method. This method takes a list of Node objects or a Document as input.


```Python
index = VectorStoreIndex.from_documents(documents)
```

If connected to an external vector database, Llama Index follows the index algorithm of that vector database. If there is no connection to an external vector database, it stores the data in the local memory in the form of a dictionary.

In [25]:
### YOUR CODE HERE ###

index = VectorStoreIndex.from_documents(documents)

Now, let’s look at how the index is structured.  

Our index is built based on documents. However, as mentioned in the previous presentation, when Llamaindex stores data in the database, it breaks down the document into nodes. Therefore, let’s first check if the document has been divided into nodes.

In [26]:
### YOUR CODE HERE ###

node_id = index.index_struct.nodes_dict

for key, value in node_id.items():
    print(key, value)
    node_example_id = value
    break

print("The number of nodes: ", len(node_id.values()))

print(node_id.values())

3e3d5097-6199-40f8-ab45-26a36f5ecd47 3e3d5097-6199-40f8-ab45-26a36f5ecd47
The number of nodes:  22
dict_values(['3e3d5097-6199-40f8-ab45-26a36f5ecd47', '1979d3db-dfb7-44ac-9be9-0f8958a13e74', 'c949685c-d7b8-4211-a1ac-999098c0d1e0', '9216b213-89bb-40c6-a4f8-1528794cfec0', '661c629c-ae7c-435e-b89c-57b624077cf7', '6f51b9bb-39ae-4b62-aa67-994909b4b246', 'fd4f48e8-5247-4c03-bd72-5057a26b38ad', '915a921b-79cb-4758-b7ab-65ea1522b6f4', '6217c963-3b05-4279-bddd-77b19a6292e6', '43a8ff9d-fe81-473b-9c8a-3e17513eb654', '22abd80c-19e3-49bf-81d7-8377314e4261', '2e7e9426-37a2-4269-83a9-2211aa4b5463', 'c1807443-96a6-4f87-bf0e-d2d998a8bdca', '6af2b9c4-008b-4980-861f-aae51529a935', '39179452-b3a3-4bb2-94ec-9cb4c1d9ca0a', 'd0949203-a91d-4fdb-8b5d-403449478b65', '3192a283-2374-4075-849a-fa63880aa8ae', 'fa67dfbd-647c-40fe-b249-57b6ee4698ec', '6d5b1fe3-de74-4880-b8c6-5f23ae946fd8', '138fdb39-baf7-403a-9a37-3d09c38fdd80', '5c96bf1d-4546-4c6d-ae39-2ef5fb6ef290', 'd9e41548-8d84-4319-9898-18d236010776'])


By looking at this, we can check the id of the first node and the number of nodes the document has been divided into.

Next, let’s see how each node has been transformed into embeddings.


In [29]:
### YOUR CODE HERE ###

index.vector_store.data.embedding_dict[node_example_id]
len(index.vector_store.data.embedding_dict[node_example_id])

1536

As you can see here, each node is automatically converted into embeddings, becoming vectors when the index is created.

Next, let’s verify why the total number of nodes is as shown. When splitting a document into nodes, Llamaindex uses a class called SentenceSplitter. This class allows us to divide the text data into predefined lengths.

In [34]:
### YOUR CODE HERE ###

from llama_index.core.node_parser import SentenceSplitter

parser = SentenceSplitter(chunk_size=1024, chunk_overlap=200)
# you can change chunk_size, chunk_overlap

nodes = parser.get_nodes_from_documents(documents)

print(len(nodes))

22


Each node has an attribute called `text`, allowing us to see what text data each node contains. Additionally, you can see that the texts overlap by the value of the `chunk_overlap`.

When checking the length of each text data, we can see that they are composed of different numbers of characters. This is because the `SentenceSplitter` class divides `documents` not by the number of characters but by tokens, which are a type of sub-word.

In [33]:
### YOUR CODE HERE ###

print(nodes[0].text)
print("\n----------------------\n")
print(nodes[1].text)
print("\n----------------------\n")
print(len(nodes[0].text), len(nodes[1].text))

What I Worked On

February 2021

Before college the two main things I worked on, outside of school, were writing and programming. I didn't write essays. I wrote what beginning writers were supposed to write then, and probably still are: short stories. My stories were awful. They had hardly any plot, just characters with strong feelings, which I imagined made them deep.

The first programs I tried writing were on the IBM 1401 that our school district used for what was then called "data processing." This was in 9th grade, so I was 13 or 14. The school district's 1401 happened to be in the basement of our junior high school, and my friend Rich Draves and I got permission to use it. It was like a mini Bond villain's lair down there, with all these alien-looking machines — CPU, disk drives, printer, card reader — sitting up on a raised floor under bright fluorescent lights.

The language we used was an early version of Fortran. You had to type programs on punch cards, then stack them in the

We can use the default tokenizer used by Llamaindex’s `SentenceSplitter` to directly check how many tokens each text consists of.

By running the code below, we can bring in the tokenizer of `gpt-4o-mini`, which is used by default in `SentenceSplitter`, and perform tokenization to see the results and check how many tokens are generated.

When we look at the tokenization results, we will see that they differ from the `chunk_size` we set. This is because, when creating nodes, the amount of text data to be tokenized is determined by considering the metadata as well, in order to compose nodes of consistent size including the amount of metadata.

In [58]:
### YOUR CODE HERE ###

import tiktoken
tokenizer = tiktoken.encoding_for_model("gpt-4o-mini")

encoded_text = tokenizer.encode(nodes[0].text)
print(f"Encoded text: {encoded_text}")
print(f"Encoded text length: {len(encoded_text)}")

decoded_text = tokenizer.decode(encoded_text)
print(f"Decoded text: \n\n{decoded_text}")

Encoded text: [4827, 357, 197609, 2160, 279, 35728, 220, 1323, 16, 279, 13036, 8730, 290, 1920, 2758, 3283, 357, 9183, 402, 11, 7539, 328, 3474, 11, 1504, 5281, 326, 23238, 13, 357, 9289, 5067, 25779, 13, 357, 11955, 1412, 10526, 20461, 1504, 18126, 316, 5067, 1815, 11, 326, 7038, 2928, 553, 25, 4022, 9970, 13, 3673, 9970, 1504, 44421, 13, 3164, 1458, 35509, 1062, 12857, 11, 1327, 9862, 483, 5532, 22353, 11, 1118, 357, 58224, 2452, 1373, 8103, 364, 976, 1577, 8094, 357, 10471, 5281, 1504, 402, 290, 40198, 220, 10165, 16, 484, 1039, 3474, 14309, 2061, 395, 1412, 673, 1815, 4358, 392, 1074, 12323, 3692, 1328, 673, 306, 220, 24, 404, 15349, 11, 813, 357, 673, 220, 1311, 503, 220, 1265, 13, 623, 3474, 14309, 885, 220, 10165, 16, 12570, 316, 413, 306, 290, 41657, 328, 1039, 38121, 1932, 3474, 11, 326, 922, 5168, 12436, 3808, 6808, 326, 357, 3508, 14158, 316, 1199, 480, 13, 1225, 673, 1299, 261, 15065, 34188, 83384, 885, 557, 380, 1917, 1354, 11, 483, 722, 1879, 38726, 43721, 14656, 2733, 28

You can see that the result here matches the number we saw earlier.  

If you want to change the default `chunk_size` or `chunk_overlap` when you make `index`, you can use `transformations` argument in `VectorStoreIndex.from_documents()`.


In [36]:
### YOUR CODE HERE ###

text_splitter = SentenceSplitter(chunk_size=200, chunk_overlap=50)

index = VectorStoreIndex.from_documents(documents=documents, transformations=[text_splitter])

In [39]:
len(index.index_struct.nodes_dict)

157

In [40]:
### YOUR CODE HERE ###

node_id = index.index_struct.nodes_dict

for key, value in node_id.items():
    print(key, value)
    node_example_id = value
    break

print("The number of nodes: ", len(node_id.values()))

print(node_id.values())

bde15477-b774-4449-9d58-67cc14147c16 bde15477-b774-4449-9d58-67cc14147c16
The number of nodes:  157
dict_values(['bde15477-b774-4449-9d58-67cc14147c16', '83c9665a-8082-4e74-9272-badf99438053', '53b6ef68-a1df-4a70-b29a-7403dd3de535', '4e6784a8-6dd3-4a89-bf93-10052d3e4b0c', '9e346048-8177-4a42-ade3-fb8bb46b5dbf', 'ff1ea8bf-5630-4b7e-8444-6abc8c737c95', 'df73aaa3-7785-4fce-a92a-27649e5d6206', '0d2f31ce-9ec0-489c-ba04-e6812aff5eaf', '902237d5-2fed-4649-a96e-a1ff99a3af24', 'a05aaa0a-b313-4f21-8f4e-7daac103e24d', 'd58cc2ad-011e-477e-9ecc-747f29c954bd', 'a4e185a4-9433-4dfe-810b-be1dee60708c', '2e282697-538d-47dc-b312-990e2d9f736f', '866bf47f-1928-4e3d-a119-2ac1de2d6e36', '1510df28-8e6a-403e-b5e6-f10721d9e470', '3756c3dd-42b1-4072-8642-d94d1a918eb9', 'af21af88-e87d-4d3f-a021-f3d04ef52eef', 'd60fa574-a0e4-48a4-aef7-fc5041ab44f9', 'c0dfc505-5ea2-4b1c-8cdf-14fa0aab0ec9', 'a96130be-c47d-43e2-bc63-bec969d0720e', 'fa9cc430-c85e-4425-aee9-fda6dcfe0b26', '5f5638f9-561e-45f7-bfba-94e9b03ced0b', 'e8080a

But, in this practice, we will use default `SentenceSplitter`.

In [41]:
### YOUR CODE HERE ###

index = VectorStoreIndex.from_documents(documents)

In [42]:
len(index.index_struct.nodes_dict)

22

Once the index is created, Llama Index provides a way to immediately create a query engine using the index. With the following code, we can create a query engine using the index.

In [52]:
### YOUR CODE HERE ###
from llama_index.llms.openai import OpenAI as LlamaOpenAI
llm = LlamaOpenAI(model="gpt-4o-mini", temperature=0)

query_engine = index.as_query_engine(llm=llm)

In [44]:
print(f'{query_engine}')

### 4. Conduct Activity

Now let's actually use the query engine. We will check the results of query engine using the following questions.

#### 4-1. Answerable Questions from Data

In this activity, we will check how query engine will answer **a question related to the given data, and the information needed for the correct answer is directly provided in the data**.

#### 4-2. Unrelated Question to the Data

In this activity, we will check how query engine will answer **a question that has absolutely nothing to do with the given data**.

#### 4-3. Question Requiring Complex Reasoning

In this activity, we will check how query engine will answer **a question that is hard to answer, which means that question is related to the given data, but the information needed for the correct answer is not directly given in the data**.

### 4-1. Answerable Questions from Data
First, let's ask the query engine some questions that it can answer and some that it cannot.

For example, we could ask what program the author of this data has written about. The answer to this question is in the second sentence of the txt file.

#### Question: **What is the first programs the author tried writing?**

#### In paul_graham_essay: **The first programs I tried writing were on the IBM 1401 that our school district used for what was then called "data processing."**

Intuitively, it is expected that the query engine will generate the correct answer, which is IBM 1401. Let's verify that.

In [45]:
### YOUR CODE HERE ###

text_chunk = "The first programs I tried writing were on the IBM 1401 that our school district used for what was then called \"data processing.\""

if text_chunk in documents[0].text:
    print("True")
else:
    print("False")

True


In [53]:
### YOUR CODE HERE ###

response = query_engine.query("What is the first programs the author tried writing?", )

print(response)

The first programs the author tried writing were on the IBM 1401, using an early version of Fortran.


<img src="https://xe.obg.co.kr/files/attach/images/4199/352/004/7e2189cf45a27f9b9cda5fef28c1dd5f.gif">

As expected, the results are accurate.  


### 4-2. Unrelated Question to the Data

This time, We will check whether the query engine performs its intended function well.  
A query engine is **a generic interface that allows users to ask questions about external data**.  
Therefore, **it should not be able to answer questions that cannot be answered through external data**.

The reason is that if the query engine can answer questions that are completely unrelated to the external data, it implies that the query engine can freely use parameterized knowledge. This, in turn, means that the answers from the query engine may contain information that is incorrect or not up-to-date.

<img src="https://media0.giphy.com/media/ANbD1CCdA3iI8/200w.gif?cid=6c09b952ooe7fzryithl047ty5npdt1xd50tlu9gcpqnzh87&ep=v1_gifs_search&rid=200w.gif&ct=g" width="200" height="200"><img src="https://st2.depositphotos.com/4421345/11492/v/950/depositphotos_114921728-stock-illustration-public-speaking-robot.jpg" width="200">

So, let’s pass on a question that has nothing to do with the original data but that the query engine’s LLM can answer.  
For example, we could use a question like this.

#### Question: **How many countries participated in the production of the space station?**
#### Answer: **15**

In [55]:
### YOUR CODE HERE ###

response = query_engine.query("How many countries participated in the production of the space station?")

print(response)

The provided information does not mention anything about the production of a space station or the countries involved in it. Therefore, I cannot provide an answer to that query.


<img src = "https://img.danawa.com/images/descFiles/6/164/5163558_1_16652362092043267.gif
" width = 200>

It doesn't produce the correct answer as expected. However, we can find what kinds of LLM used in query engine through the official LlamaIndex documentation.

Actually, Llamaindex uses `gpt-4o-mini` as LLM of response synthesizer. So, we can ask `gpt-4o-mini` to answer the question.

Let's check this out with a simple code. We can use the following code to separately utilize the OpenAI LLM.  



In [59]:
### YOUR CODE HERE ###

def generate_answer(question):
    messages = [
        {
            "role": "user",
            "content": question,
        },
    ]
    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=messages,
    )

    return response.choices[0].message.content

In [60]:
### YOUR CODE HERE ###

question = "How many countries participated in the production of the space station?"

print(generate_answer(question))

The International Space Station (ISS) is a collaborative project involving multiple countries. The primary partners in the ISS program are:

1. United States (NASA)
2. Russia (Roscosmos)
3. Europe (European Space Agency - ESA, which includes 22 member countries)
4. Japan (JAXA)
5. Canada (Canadian Space Agency - CSA)

In total, the ISS program involves contributions from 15 countries, including the primary partners and the member countries of ESA. The collaboration encompasses various aspects of the station's design, construction, and operation.


In fact, as in the example above, OpenAI LLM already knows the answer to this question.  
However, since the relevant information cannot be found in the txt file that query engine can refer to here, it does not generate a correct answer for a question that is already known.  

<img src="https://i.imgur.com/sxSRQks.png">

From this, we can understand that LLM in the response synthesizer of the query engine uses the information from the retrieved passages produced by the retriever, excluding its own parametrized knowledge.  

### 4-3. Question Requiring Complex Reasoning

Now, let’s explore more deeply. There is a question related to the data that is hard to answer. We will see how the query engine responds to such questions.

The question we will ask in this section is who is the author of the `paul_graham_essay.txt`. Becuase the original text file does not mention the author's name directly, this is a complex question.

Question: **Who is the author?**

Let's see whether the writer's name is actually not in the original text file or not.

In [61]:
### YOUR CODE HERE ###

wrapper = textwrap.TextWrapper(width=80)

with open(path_to_txt, 'r', encoding='utf-8') as file:
    for line in file:
        if 'paul graham' in line.lower():
            formatted_text = wrapper.fill(line.strip())
            print(formatted_text.split('.')[-1])


Occasionally after wrestling for hours with some gruesome bug I'd check Twitter
or HN and see someone asking "Does Paul Graham still code?"


This is the only sentence in the original txt file that mentions 'Paul Graham'.  
Now, let's read it ourselves and consider whether it actually specifies the author's name.

<br/>

Someone might infer that the author's name is Paul Graham from this sentence, but I think it's not certain

In my opinion, this sentence does not explicitly state the author's name, so it does not provide enough useful information

<br/>

However, opinions on this may vary, so let's verify it ourselves.


In [62]:
### YOUR CODE HERE ###

response_complex = query_engine.query("Who is the author?")

print(response_complex)

The author is Paul Graham.


<img src="https://i3.ruliweb.net/ori/21/07/30/17af799f631524cc2.gif">

Hmm... The query engine answers better than we think.  
Then, let's see what data were referred to by the query engine.
<br/>

<img src="https://i.imgur.com/sxSRQks.png">

The query engine made using `as_query_engine` uses a retriever generated directly from the index by using `as_retriever` method.   
Therefore, if we look at the results of this retriever, we can see which retrieved passages the query engine has seen.


In [63]:
### YOUR CODE HERE ###

retriever = index.as_retriever()

ret_passages = retriever.retrieve("Who is the author?")

for i in range(len(ret_passages)):
  print("###Retrieved Passage\n", ret_passages[i].text)
  print("\n\n\n")

###Retrieved Passage
 Except for a few officially anointed thinkers who went to the right parties in New York, the only people allowed to publish essays were specialists writing about their specialties. There were so many essays that had never been written, because there had been no way to publish them. Now they could be, and I was going to write them. [12]

I've worked on several different things, but to the extent there was a turning point where I figured out what to work on, it was when I started publishing essays online. From then on I knew that whatever else I did, I'd always write essays too.

I knew that online essays would be a marginal medium at first. Socially they'd seem more like rants posted by nutjobs on their GeoCities sites than the genteel and beautifully typeset compositions published in The New Yorker. But by this point I knew enough to find that encouraging instead of discouraging.

One of the most conspicuous patterns I've noticed in my life is how well it has work

This is quiet lone passages. So, it is hard to check whether there is any evidence of query engine's response or not.  
Can you find any supporting evidence of the question?  

Let's see if 'Paul Graham' is in there.



In [64]:
### YOUR CODE HERE ###

passages = ''

for i in range(len(ret_passages)):
  passages += ret_passages[i].text

if 'paul graham' in passages.lower():
    print("The word 'Paul Graham' is found in the passages.")
else:
    print("The word 'Paul Graham' is not found in the passages.")

The word 'Paul Graham' is not found in the passages.


<img src="https://i2.ruliweb.net/ori/21/07/24/17ad8e756ba54811a.gif" width="400">

???

Retriever couldn't find the only sentence with the word Paul Graham on it. Then how did query engine know that the answer was Paul Graham?

To verify this, let’s using the query engine’s LLM as we did in the previous section and ask the same question. We will use a prompt as similar as possible to the one used in LlamaIndex response synthesizer, but with an additional instruction to provide the reason for the LLM’s answer.  

The prompt used by the original Response Synthesizer is as follows.


<img src = "https://i.imgur.com/ZXUDAJi.jpeg" width="450">  


<br/>

Therefore, we will write the code as shown below to check the results of the LLM.



In [65]:
### YOUR CODE HERE ###

ret_context = ""
for ret_result in ret_passages:
  ret_context += ret_result.text

question = f"""Context information is below.
---------------------
{ret_context}
---------------------
Given the context information and not prior knowledge,
answer the query and the reasons of answer.
Query: Who is the author?
Answer:
"""

print(generate_answer(question))

The author is Paul Graham. 

**Reasons for the answer:**

1. **Contextual Clues**: The text discusses the author's experiences with writing essays, starting companies, and his involvement in the tech startup scene, particularly mentioning Y Combinator, which he co-founded. Paul Graham is well-known for his essays on startups and technology.

2. **Personal Experiences**: The narrative includes personal anecdotes and reflections that align with Paul Graham's known biography, such as his early ventures in software development and his thoughts on the nature of work and prestige.

3. **Specific References**: The mention of "Hackers & Painters," a book by Paul Graham that compiles his essays, further supports the identification of the author.

4. **Style and Themes**: The writing style and themes of entrepreneurship, innovation, and the evolution of technology are characteristic of Paul Graham's work. 

Overall, the combination of these elements strongly indicates that the author is Paul Gra

Well... it seems like query engine can **capture useful information not only in direct evidence but also in indirect content by using its prior knowledge, i.e., parametrized knowledge**.  

Therefore, we can consider that the query engine has the ability to provide correct answers even when the given query is not explicitly answered in the data.  

However, the problem is that **without such an explanation, the reasoning process cannot be trusted** because it is impossible to know whether the LLM used reliable information in its reasoning.

<br/>

Here’s what we can infer from this:

the query engine has the capability to answer not only questions directly addressed in the data **but also questions that can be indirectly inferred from the data.**  
However, in such cases, **the reliability of the answers is compromised because we cannot verify the information used and the reasoning steps.**   

To address this issue, we can use one of the following two methods.  


#### **1.** Force the response synthesizer to utilize only the information in a given retrieved passages.
#### **2.** When the response synthesizer generates a summary, make sure that the reason is also generated.

## II. Use Database Management Method

Now, let’s consider a scenario. If we have new data and want to create a query engine using it, we could recreate the `document`, `index`, and `query engine` as we did above.

However, this is quite cumbersome and time-consuming. In addition, if you want to insert new data so that query engine uses that data, the above method is very expansive because a new index must be generated for every inserts operation.

So, Llama Index is providing the following functions to manage data smoothly.

### 1. Insert

We can insert new data to the existing query engine.

To check this, we will ask a question which can't get useful information from the data before inserting new data, and see what happens after inserting data.

In this section, the following data and question will be used.

Inserted Data: **"Natural diamonds were (and are) formed (thousands of million years ago) in the upper mantle of Earth in metallic melts at temperatures of 900–1,400 °C and at pressures of 5–6 GPa."**  
Question: **What are the temperature and air pressure conditions under which natural diamonds are produced?**


In [66]:
### YOUR CODE HERE ###

data_text = "Natural diamonds were (and are) formed (thousands of million years ago) in the upper mantle of Earth in metallic melts at temperatures of 900–1,400 °C and at pressures of 5–6 GPa."
question = "What are the temperature and air pressure conditions under which natural diamonds are produced?"

res = query_engine.query(question)

print(res)

The provided information does not include any details about the temperature and air pressure conditions under which natural diamonds are produced.


<img src = "https://i.pinimg.com/originals/15/8b/ed/158bed9819e4fccf7e18a5eeeaf79c6b.png" width = 200>

Hmm… the query engine answers quite well. It seems the response synthesizer is using its parameterized knowledge. However, there are no specific details like 900–1,400°C and pressures of 5–6 GPa.

Anyway, it's time to insert the data.



In [68]:
### YOUR CODE HERE ###

docu = Document(text=data_text, id_="new_doc_id")

index.insert(docu)

new_query_engine = index.as_query_engine(llm=llm)

In [69]:
### YOUR CODE HERE ###

new_res = new_query_engine.query(question)

print(new_res)

Natural diamonds are produced at temperatures of 900–1,400 °C and at pressures of 5–6 GPa.


It's as expected. Query engine is generating an appropriate answer with detail using information in new `Document` object.

Then, what happens if we change the information in that data?

### 2. Update

If a Document is already present within an index, you can "update" a `Document` with the same doc `id_`. The `update_ref_doc` method receives a single `Document` object and updates the value of data that has the same `id`.

Or you can "refresh" all document at once. The `refresh_ref_doc` method finds the `id` of the `Document` object that came into the input, updates the data with the same `id`, or inserts the data if there is no other data with the same `id`

**`update_ref_doc`**  
**Input**: single `Document` object  
**Output**: None

**`refresh_ref_doc`**  
**Inpu**t: list of `Document` object  
**Output**: a boolean list, indicating which documents in the input have been refreshed in the index.

You will see `False` in that boolean list if text of document doesn't change.  

In both methods, we can set `delete_from_docstore` to `True` or `False`. A detailed description of this will be given in Delete.


<br/>

Let's check by changing the base question a little bit.  

I changed the information about temperature and air pressure in the question, so check it out for yourself.  



In [70]:
### YOUR CODE HERE ###

docu.set_content(value="Natural diamonds were (and are) formed (thousands of million years ago) in the upper mantle of Earth in metallic melts at temperatures of 2,000–6,000 °C and at pressures of 8–9 GPa.")

output = index.update_ref_doc(
    docu,
    update_kwargs={"delete_kwargs": {"delete_from_docstore": True}},
)

print(output)

query_engine_update = index.as_query_engine(llm=llm)

res_update = query_engine_update.query(question)

print(res_update)

None
Natural diamonds are formed at temperatures ranging from 2,000 to 6,000 °C and at pressures of 8 to 9 GPa.


In [71]:
### YOUR CODE HERE ###

docu.set_content(value="Natural diamonds were (and are) formed (thousands of million years ago) in the upper mantle of Earth in metallic melts at temperatures of 6,000–8,000 °C and at pressures of 12–15 GPa.")

output = index.refresh_ref_docs(
    [docu]
)

print(output)

query_engine_refresh = index.as_query_engine(llm=llm)

res_refresh = query_engine_refresh.query(question)

print(res_refresh)

[True]
Natural diamonds are produced at temperatures ranging from 6,000 to 8,000 °C and at pressures between 12 to 15 GPa.


You can check list of doc_id in the index. So, if you want more experience with other data, you can check whether your data is inserted properly by looking at that list

In [72]:
### YOUR CODE HERE ###

print(index.ref_doc_info.keys())

dict_keys(['28131446-24ce-47c7-87bd-a7f0f51df1e1', 'new_doc_id'])


### 3. Delete

We wrapped the new information above with a `Document` object and inserted it into the index.

At this time, we were able to specify the `Document` object using `doc_id`. What this doc_id is can be confirmed as follows.

In [73]:
### YOUR CODE HERE ###

id = docu.doc_id

print(id)

new_doc_id


Using this, we can change the contents of data with the `doc_id` using the `delete` method of the index.

Let's delete the data related to a diamond, and see how query engine answers for the same question.

Through the value of `delete_from_docstroe`, it is possible to determine whether the data with that id actually disappears on the database, or disappears only on the index and remains in the database. Even if it disappears only on the index, the information of the data cannot be used by the query engine.



In [74]:
### YOUR CODE HERE ###

index.delete_ref_doc(id, delete_from_docstore=True)

query_engine_delete = index.as_query_engine(llm=llm)

res_delete = query_engine_delete.query(question)

print(res_delete)

The provided information does not include any details about the temperature and air pressure conditions under which natural diamonds are produced.


If the answer still contains very details of condition of natural diamonds, check the index whether that data was deleted properly.

The result of `index.ref_doc_info.keys()` has to contain only one doc_id.

In [75]:
### YOUR CODE HERE ###

print(index.ref_doc_info.keys())

dict_keys(['28131446-24ce-47c7-87bd-a7f0f51df1e1'])


## III. Make a Custom Query Engine

In this section, we will define the existing query engine in more detail and modify its internal components to create the query engine we desire.
<br/>

The query engine can vary in its usage depending on the task requested after passing the retrieved documents to the LLM in response synthesizer. For instance, it can be tasked with summarizing the documents or directly answering the query.

Since the query engine can read through extremely long documents that exceed the LLM’s context limit, it can have a significant impact on RAG, depending on how it is used. Therefore, choosing which query engine to use is an important design choice when designing RAG.

This section is composed of the following four stages:
 <br/>
#### 1. Defining the query engine in detail
#### 2. Creating a custom query engine


### 1. Defining the query engine in detail

In this section, we will define the retriever and response synthesizer, which are internal components of the query engine, and use them to define a class that performs the role of the query engine.

In [76]:
### YOUR CODE HERE ###

class StandardQueryEngine(CustomQueryEngine):
    retriever: BaseRetriever
    response_synthesizer: BaseSynthesizer

    def custom_query(self, query_str: str):
        nodes = self.retriever.retrieve(query_str)
        response_obj = self.response_synthesizer.synthesize(query_str, nodes)
        return response_obj

Initialize the retriever and response synthesizer, and then verify that the class functions correctly.


In [77]:
### YOUR CODE HERE ###

retriever = index.as_retriever()
synthesizer = get_response_synthesizer(response_mode="compact", llm=llm)
query_engine = StandardQueryEngine(
    retriever=retriever, response_synthesizer=synthesizer
)

response = query_engine.query("What did the author do growing up?")
print(str(response))

Growing up, the author focused on writing and programming outside of school. Initially, he wrote short stories, which he later described as lacking in plot but filled with characters expressing strong feelings. He began programming at the age of 13 or 14 on an IBM 1401, where he experimented with an early version of Fortran, although he found it puzzling and struggled to create meaningful programs. Eventually, he got a TRS-80 microcomputer, which allowed him to write simple games and a word processor used by his father. Despite enjoying programming, he initially planned to study philosophy in college, believing it to be a pursuit of ultimate truths.


### 2. Creating a custom query engine

Now, let’s modify the query engine so that we can pass the desired task to the response synthesizer through a prompt.  

While there are many aspects of the retriever that can also be adjusted, in this exercise, we will focus on changing the content passed to the response synthesizer to experience more noticeable changes

from llama_index.core import PromptTemplate


In [78]:
### YOUR CODE HERE ###

simple_qa_prompt = PromptTemplate(
    "Context information is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Given the context information and not prior knowledge, "
    "answer the query.\n"
    "Query: {query_str}\n"
    "Answer: "
)

short_sum_prompt = PromptTemplate(
    "Context information is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Given the context information and not prior knowledge, "
    "give the summary of context, less than 20 words.\n"
    "Query: {query_str}\n"
    "Summary: "
)

In [81]:
### YOUR CODE HERE ###

from llama_index.llms.openai import OpenAI

class OurCustomQueryEngine(CustomQueryEngine):

    retriever: BaseRetriever
    response_synthesizer: BaseSynthesizer
    llm: OpenAI
    qa_prompt: PromptTemplate = simple_qa_prompt

    def custom_query(self, query_str: str):
        nodes = self.retriever.retrieve(query_str)

        context_str = "\n\n".join([n.node.get_content() for n in nodes])
        response = self.llm.complete(
            self.qa_prompt.format(context_str=context_str, query_str=query_str)
        )

        return str(response)

Now, let’s check if it truly works as we intended.


In [83]:
### YOUR CODE HERE ###

#llm = OpenAI(model="gpt-4o-mini")
#llm=llm

query_engine = OurCustomQueryEngine(
    retriever=retriever,
    response_synthesizer=synthesizer,
    llm=llm,
    qa_prompt=short_sum_prompt,
)

response = query_engine.query("What did the author do growing up?")
print(str(response))

The author wrote short stories and programmed on early computers, eventually transitioning to AI and essay writing.
